# Model Compression & Quantization for Edge Deployment of a Keyword-Spotting Model
Model Compression & Quantization for Edge Deployment of a Audio Keyword Spotting Model

**Mini-Project — Introduction to Machine Learning Lab**

## 1. Introduction & Problem Setup

Always-on voice interfaces (wake-word detection, short-command recognition) run on
power- and memory-constrained edge devices — microcontrollers, DSPs, or low-power SoCs —
where a full floating-point deep learning model is often too large or too slow to run
continuously. This project studies **model compression via quantization** applied to a
small convolutional neural network trained for **keyword spotting (KWS)** on the
Google Speech Commands dataset.

**Goals of this notebook:**
1. Train a small CNN baseline (FP32) on a subset of spoken keywords.
2. Apply post-training quantization (PTQ) — and optionally quantization-aware
   training (QAT) — to compress the model to INT8.
3. Export both FP32 and INT8 models to ONNX and verify correctness.
4. Quantitatively compare **accuracy, model size, and inference latency** across
   all variants, and discuss the accuracy-efficiency trade-off.

This mirrors real edge-AI deployment workflows (e.g. TensorFlow Lite / ONNX Runtime
pipelines used in mobile and embedded audio processing).

> **TODO (you):** Fill in your name, roll number, course code, and instructor name here.


## 0. Environment Setup

Run this once. If a package is missing, install it in a separate cell with `!pip install <package>`.

In [ ]:
# Core libraries
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import torchaudio
from torchaudio.datasets import SPEECHCOMMANDS

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Data Loading & Preprocessing

We use **Google Speech Commands v2**, downloaded automatically via `torchaudio`.
To keep training fast on a laptop/Colab, we restrict to a **subset of keywords**
(a deliberate scoping choice — mention this in your report's methodology section).

Each 1-second waveform is converted to a **log-Mel spectrogram**, which is the
standard input representation for audio CNNs (this feature-extraction step is itself
a signal-processing component worth describing in your report).


In [ ]:
# --- Choose your keyword subset here ---
# Full label set has 35 words; we pick a manageable subset (edit as you like)
SELECTED_LABELS = ["yes", "no", "up", "down", "left", "right",
                    "on", "off", "stop", "go"]
LABEL_TO_IDX = {label: i for i, label in enumerate(SELECTED_LABELS)}
NUM_CLASSES = len(SELECTED_LABELS)
print(f"Using {NUM_CLASSES} keywords: {SELECTED_LABELS}")


In [ ]:
# --- Download dataset (first run only; cached afterwards) ---
DATA_ROOT = "./data"
os.makedirs(DATA_ROOT, exist_ok=True)

train_raw = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="training")
val_raw   = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="validation")
test_raw  = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="testing")

print("Train / Val / Test raw sizes:", len(train_raw), len(val_raw), len(test_raw))


In [ ]:
SAMPLE_RATE = 16000
N_MELS = 40
FIXED_LENGTH = SAMPLE_RATE  # pad/truncate every clip to exactly 1 second

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=400, hop_length=160, n_mels=N_MELS
)
db_transform = torchaudio.transforms.AmplitudeToDB()

def waveform_to_features(waveform):
    """Pad/truncate waveform to FIXED_LENGTH samples, then convert to log-Mel spectrogram."""
    if waveform.shape[1] < FIXED_LENGTH:
        pad = FIXED_LENGTH - waveform.shape[1]
        waveform = F.pad(waveform, (0, pad))
    else:
        waveform = waveform[:, :FIXED_LENGTH]
    mel = mel_transform(waveform)
    mel_db = db_transform(mel)
    return mel_db  # shape: [1, N_MELS, time_frames]


class KeywordSpottingDataset(Dataset):
    """Wraps a torchaudio SPEECHCOMMANDS split, keeping only SELECTED_LABELS."""

    def __init__(self, raw_dataset, label_to_idx):
        self.label_to_idx = label_to_idx
        self.items = [
            i for i in range(len(raw_dataset))
            if raw_dataset[i][2] in label_to_idx
        ]
        self.raw_dataset = raw_dataset

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        waveform, sample_rate, label, *_ = self.raw_dataset[self.items[idx]]
        features = waveform_to_features(waveform)
        target = self.label_to_idx[label]
        return features, target


train_ds = KeywordSpottingDataset(train_raw, LABEL_TO_IDX)
val_ds   = KeywordSpottingDataset(val_raw, LABEL_TO_IDX)
test_ds  = KeywordSpottingDataset(test_raw, LABEL_TO_IDX)

print("Filtered Train / Val / Test sizes:", len(train_ds), len(val_ds), len(test_ds))

# Peek at one sample's feature shape
sample_feat, sample_label = train_ds[0]
print("Feature shape (channels, n_mels, time_frames):", sample_feat.shape)


In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


## 3. Baseline Model (FP32)

A small CNN over the log-Mel spectrogram (treated as a 2D "image": time x frequency).
Kept deliberately small (a few conv layers, well under 1M parameters) — the point of
this project is **compressibility and edge-readiness**, not chasing state-of-the-art
accuracy.


In [ ]:
class KWSNet(nn.Module):
    def __init__(self, n_classes, n_mels=N_MELS):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # Infer flattened feature size dynamically the first time we run a forward pass
        self._flat_dim = None
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.LazyLinear(n_classes)  # infers input size automatically on first forward
        )

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = KWSNet(NUM_CLASSES).to(device)

# Initialize LazyLinear by running one dummy batch through the model
with torch.no_grad():
    dummy = torch.zeros(1, *sample_feat.shape).to(device)
    _ = model(dummy)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameter count: {n_params:,}")
print(model)


## 4. Training & Baseline Evaluation

Standard train/validation loop. We log training/validation accuracy per epoch, then
report **baseline accuracy, model size (MB), and CPU inference latency (ms/sample)**
— these three numbers are the reference point for every quantized variant that follows.


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for features, targets in loader:
            features, targets = features.to(device), targets.to(device)
            outputs = model(features)
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return correct / total


def train(model, train_loader, val_loader, device, epochs=10, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_acc": [], "val_acc": [], "train_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * targets.size(0)
            correct += (outputs.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)

        train_acc = correct / total
        train_loss = running_loss / total
        val_acc = evaluate(model, val_loader, device)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_loss"].append(train_loss)

        print(f"Epoch {epoch:2d}/{epochs} | loss: {train_loss:.4f} | "
              f"train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f}")

    return history


EPOCHS = 10  # TODO: tune based on how quickly validation accuracy plateaus
history = train(model, train_loader, val_loader, device, epochs=EPOCHS)


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
import io

def model_size_mb(model):
    """Serialize state_dict to an in-memory buffer and measure its size."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return len(buf.getvalue()) / (1024 ** 2)


def measure_latency(model, sample_input, device, n_runs=50):
    """Average CPU inference latency in milliseconds over n_runs, after a few warm-up runs."""
    model.eval()
    model.to(device)
    sample_input = sample_input.to(device)
    with torch.no_grad():
        for _ in range(5):  # warm-up
            _ = model(sample_input)
        start = time.perf_counter()
        for _ in range(n_runs):
            _ = model(sample_input)
        end = time.perf_counter()
    return (end - start) / n_runs * 1000  # ms


cpu_device = torch.device("cpu")
model_cpu = KWSNet(NUM_CLASSES)
model_cpu.load_state_dict(model.state_dict())
model_cpu.to(cpu_device)

baseline_acc = evaluate(model, test_loader, device)
baseline_size = model_size_mb(model)
baseline_latency = measure_latency(model_cpu, dummy.cpu(), cpu_device)

print(f"[FP32 Baseline] Test accuracy: {baseline_acc:.4f} | "
      f"Size: {baseline_size:.3f} MB | Latency: {baseline_latency:.3f} ms")


## 5. Quantization

This is the core of the project. We start with **dynamic post-training quantization**
(simplest, no calibration data needed), then extend to **static PTQ with calibration**
and, time permitting, **quantization-aware training (QAT)**. Comparing at least two of
these techniques is what turns this into a real analysis rather than a single data point.

> **Note:** PyTorch's native quantization backends (`fbgemm`/`qnnpack`) run on CPU.
> Make sure evaluation/latency measurement for quantized models uses `cpu_device`.


### 5.1 Dynamic Post-Training Quantization (PTQ)

In [ ]:
import torch.quantization as tq

model_cpu.eval()
dynamic_quant_model = tq.quantize_dynamic(
    model_cpu, {nn.Linear}, dtype=torch.qint8
)

dynamic_acc = evaluate(dynamic_quant_model, DataLoader(test_ds, batch_size=BATCH_SIZE), cpu_device)
dynamic_size = model_size_mb(dynamic_quant_model)
dynamic_latency = measure_latency(dynamic_quant_model, dummy.cpu(), cpu_device)

print(f"[Dynamic PTQ INT8] Test accuracy: {dynamic_acc:.4f} | "
      f"Size: {dynamic_size:.3f} MB | Latency: {dynamic_latency:.3f} ms")


### 5.2 Static Post-Training Quantization (with calibration)

Static PTQ quantizes both weights *and* activations, which requires running a handful
of calibration batches through the model first so it can observe activation ranges.
This typically compresses further than dynamic PTQ but needs the extra calibration step.


In [ ]:
class QuantizableKWSNet(nn.Module):
    """Same architecture as KWSNet, wrapped with QuantStub/DeQuantStub for static quantization."""

    def __init__(self, n_classes, n_mels=N_MELS):
        super().__init__()
        self.quant = tq.QuantStub()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.LazyLinear(n_classes)
        )
        self.dequant = tq.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        x = self.dequant(x)
        return x


# TODO: copy trained weights from `model` into a QuantizableKWSNet instance,
# then follow the standard static-PTQ recipe:
#   1. static_model.qconfig = tq.get_default_qconfig("fbgemm")
#   2. tq.prepare(static_model, inplace=True)
#   3. run a few calibration batches through static_model (no backward pass)
#   4. tq.convert(static_model, inplace=True)
# Evaluate accuracy / size / latency the same way as for dynamic PTQ above.


### 5.3 Quantization-Aware Training (QAT) — optional, if time allows

QAT simulates quantization *during* training so the model learns weights that are
robust to the precision loss, typically closing most of the accuracy gap left by PTQ.
This is the comparison that makes for the strongest "why does QAT beat PTQ" discussion
in your report.


In [ ]:
# TODO (optional, time permitting):
#   1. qat_model.qconfig = tq.get_default_qat_qconfig("fbgemm")
#   2. tq.prepare_qat(qat_model, inplace=True)
#   3. Fine-tune qat_model for a few epochs using the existing `train()` function
#   4. tq.convert(qat_model.eval(), inplace=True)
#   5. Evaluate accuracy / size / latency exactly as above


## 6. Export & Cross-Format Comparison (ONNX)

Export both the FP32 baseline and the best quantized model to ONNX, then verify with
`onnxruntime` that the exported model's predictions match the original PyTorch model —
a short but important end-to-end sanity check.


In [ ]:
# !pip install onnx onnxruntime   # uncomment if not already installed

import onnx
import onnxruntime as ort

dummy_input = dummy.cpu()

torch.onnx.export(
    model_cpu, dummy_input, "kws_fp32.onnx",
    input_names=["input"], output_names=["output"],
    opset_version=13
)

torch.onnx.export(
    dynamic_quant_model, dummy_input, "kws_int8_dynamic.onnx",
    input_names=["input"], output_names=["output"],
    opset_version=13
)

print("Exported kws_fp32.onnx and kws_int8_dynamic.onnx")


In [ ]:
def check_onnx_matches_torch(onnx_path, torch_model, sample_input, atol=1e-2):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(sample_input).numpy()

    session = ort.InferenceSession(onnx_path)
    onnx_out = session.run(None, {"input": sample_input.numpy()})[0]

    close = np.allclose(torch_out, onnx_out, atol=atol)
    max_diff = np.abs(torch_out - onnx_out).max()
    print(f"{onnx_path}: predictions match = {close} (max abs diff = {max_diff:.5f})")
    return close


check_onnx_matches_torch("kws_fp32.onnx", model_cpu, dummy_input)
# Note: dynamic-quantized models can be less exactly comparable via allclose;
# a looser atol or a top-1-prediction-match check is fine for that one.


## 7. Results & Trade-off Analysis

Fill in the table below from the numbers printed above, then generate the
accuracy-vs-size (or accuracy-vs-latency) trade-off plot. **This section is the
core of your report** — the raw notebook results matter less than a clear-eyed
discussion of the trade-offs.


In [ ]:
results = {
    "FP32 Baseline":      {"accuracy": baseline_acc, "size_mb": baseline_size, "latency_ms": baseline_latency},
    "Dynamic PTQ INT8":   {"accuracy": dynamic_acc,   "size_mb": dynamic_size,   "latency_ms": dynamic_latency},
    # "Static PTQ INT8":  {"accuracy": ..., "size_mb": ..., "latency_ms": ...},   # TODO: fill in after 5.2
    # "QAT INT8":         {"accuracy": ..., "size_mb": ..., "latency_ms": ...},   # TODO: fill in after 5.3 (optional)
}

print(f"{'Model':<20}{'Accuracy':<12}{'Size (MB)':<12}{'Latency (ms)':<14}{'Compression x':<15}")
for name, r in results.items():
    compression = baseline_size / r["size_mb"]
    print(f"{name:<20}{r['accuracy']:<12.4f}{r['size_mb']:<12.3f}{r['latency_ms']:<14.3f}{compression:<15.2f}")


In [ ]:
names = list(results.keys())
sizes = [results[n]["size_mb"] for n in names]
accs = [results[n]["accuracy"] for n in names]

plt.figure(figsize=(6, 5))
plt.scatter(sizes, accs, s=80)
for n, x, y in zip(names, sizes, accs):
    plt.annotate(n, (x, y), textcoords="offset points", xytext=(6, 4))
plt.xlabel("Model size (MB)")
plt.ylabel("Test accuracy")
plt.title("Accuracy vs. Model Size Trade-off")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Discussion (fill in once you have all rows populated):**
- How much did accuracy drop from FP32 to INT8 (dynamic / static)?
- If you ran QAT: did it recover accuracy compared to PTQ? Why would you expect that,
  given QAT sees quantization error during training while PTQ doesn't?
- What's the practical implication for deployment: is the accuracy loss acceptable
  given the size/latency savings, for an always-on edge keyword spotter?


## 8. Conclusion & Future Work

- Summarize the key trade-off finding in 2-3 sentences.
- **Limitations**: subset of 10 keywords (not the full 35-word vocabulary), no
  background-noise robustness testing, latency measured on a laptop CPU rather than
  real embedded hardware.
- **Future work**: deploy on real edge hardware (e.g. Raspberry Pi, or a microcontroller
  via TensorFlow Lite Micro / ONNX Runtime Mobile), test noise robustness, expand to
  the full keyword set, and explore structured pruning alongside quantization.
- Connect explicitly to industry relevance: this workflow (train → quantize → export
  → benchmark) mirrors real edge-AI / audio-processing pipelines used in embedded and
  mobile SoC software teams.

---
*Report note: when you write the IEEE-format report, Sections 3-7 of this notebook map
directly onto your Methodology and Results sections — keep the notebook and report
numbers consistent.*
